In [ ]:
import sys

from langchain_ollama import ChatOllama

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_core.messages import HumanMessage, AIMessage

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)

from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
)

from langchain.chains.combine_documents import (
    create_stuff_documents_chain,
)

# ==========================================
# 1. LLM Setup
# ==========================================

print("🔄 Initializing Ollama (Llama 3.1)...")

llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60,
)

# ==========================================
# 2. Embedding Model
# ==========================================

print("🔄 Loading Embedding Model...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# ==========================================
# 3. Load ChromaDB
# ==========================================

print("🔄 Connecting ChromaDB...")

vector_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
)

retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

# ==========================================
# 4. Chat History
# ==========================================

chat_history = []

# ==========================================
# 5. Question Rewriter Prompt
# ==========================================

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Given a chat history and the latest user question,
which may reference previous conversation,
rewrite the question into a standalone question.

Do NOT answer the question.
Only rewrite it if needed.
            """,
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# ==========================================
# 6. History Aware Retriever
# ==========================================

history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt,
)

# ==========================================
# 7. QA Prompt
# ==========================================

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a helpful assistant.

Answer ONLY from the provided context.

If the answer is not available in the context,
reply with:

"I don't know based on the provided context."

Context:
{context}
            """,
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# ==========================================
# 8. Document Chain
# ==========================================

question_answer_chain = create_stuff_documents_chain(
    llm,
    qa_prompt,
)

# ==========================================
# 9. Final RAG Chain
# ==========================================

rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain,
)

# ==========================================
# 10. Ask Function
# ==========================================

def run_rag_pipeline(user_question):

    print(f"\n📝 User: {user_question}")

    response = rag_chain.invoke(
        {
            "input": user_question,
            "chat_history": chat_history,
        }
    )

    answer = response["answer"]

    # Save history
    chat_history.append(
        HumanMessage(content=user_question)
    )

    chat_history.append(
        AIMessage(content=answer)
    )

    print("\n🤖 Assistant:")
    print(answer)

    print("\n📚 Chat History Length:",
          len(chat_history))

# ==========================================
# 11. Test
# ==========================================

run_rag_pipeline(
    "Who is Abu Sufiun?"
)

run_rag_pipeline(
    "How many skills does he have?"
)

run_rag_pipeline(
    "What is his latest project?"
)